# Web Scraping

This notebook will have problems that touch on the material covered in `Lectures/Data Collection/Web Scraping with BeautifulSoup`.

In [1]:
from bs4 import BeautifulSoup
import requests
import pandas as pd
from time import sleep
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

##### 1. imdb - go to film reviews page in browser

In [ ]:
# open browser
driver = webdriver.Chrome()
    

In [ ]:
# american history x reviews
# Need to log in after visiting site to get review list for obtaining reviewer URLs
driver.get("https://www.imdb.com/title/tt0120586/reviews/?sort=review_volume%2Cdesc")
#print(title)

In [ ]:
# Load 25 more reviews 3 times
# (Can be modified to get full reviewer list easily)
for i in range(3):
    button = WebDriverWait(driver, 5).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "button.ipc-see-more__button"))
        )
    #scroll on page to reach button, otherwise click fails
    driver.execute_script("arguments[0].scrollIntoView(true);", button)
    sleep(1)        
    # Click it
    button.click()

In [11]:
soup=BeautifulSoup(driver.page_source, 'html.parser')

In [ ]:
# Previous attempt to find "more reviews" button

#my_button = driver.find_element(By.XPATH, "//span[contains(@class, 'ipc-see-more__text')]//ancestor::button")
##my_button.click()
#driver.implicitly_wait(5)
#my_button.click()

In [ ]:
# Alt wait for button to be clickable
my_button = WebDriverWait(driver, 10).until(
    EC.element_to_be_clickable((By.CSS_SELECTOR, "button.ipc-see-more__button"))
)
my_button.click()

In [13]:
#users = soup.find_all('span', class_="antialiased tracking-[0.02em] !text-sm overflow-hidden text-ellipsis whitespace-nowrap"); users
user_urls = soup.find_all('a', attrs={'data-testid' : 'author-link'}); user_urls
print(len(user_urls))

100


In [14]:
usr_links=[]
for usr in user_urls:
    try:
        uname_raw= usr.find('span',class_="antialiased tracking-[0.02em] !text-sm overflow-hidden text-ellipsis whitespace-nowrap")
        url_raw=usr.get('href').split('?')[0]
        uname=uname_raw.text.strip()
        url = "https://www.imdb.com"+url_raw+'ratings'
        #year=film.find('li',class_="ipc-inline-list__item")
        #rating has two entries, first avg then user
        #rating_text = rating[1].text.strip()
        #get rid of number before the title
        #title_text = title[0].text.split('. ', 1)[1]
        #year_text = year.text.strip()
        print(uname, url)
        usr_links.append({'username': uname, 'ratings_url':url })
    except:
        # Skip if either element is not found in this <li>
        continue

    
usr_links


planktonrules https://www.imdb.com/user/p.twhkadfwuxnnk3v3crk2lbxcda/ratings
SnoopyStyle https://www.imdb.com/user/p.jdpo6uufynnvg5phhb3jhga4ji/ratings
Kirpianuscus https://www.imdb.com/user/p.idvjk25ntslz3t4zlr4bthixra/ratings
mark.waltz https://www.imdb.com/user/p.f3fpx4l7nz7xjpaxcnmbs6mfue/ratings
Sleepin_Dragon https://www.imdb.com/user/p.nadoqibapcutzfhtliuzriogla/ratings
Leofwine_draca https://www.imdb.com/user/p.f772breweshp773tnlaavsxh5m/ratings
Hitchcoc https://www.imdb.com/user/p.bprpogbwb56iaix6ajsznizkde/ratings
paul_m_haakonsen https://www.imdb.com/user/p.jhijfk6m2bldmwgfat3yuhau5m/ratings
ma-cortes https://www.imdb.com/user/p.4ezgnj6y3mutunwrpwjnf35w24/ratings
bob the moo https://www.imdb.com/user/p.5ofp5vs6p3vaea4izrcquut2ni/ratings
classicsoncall https://www.imdb.com/user/p.a2czb7imou26utvw3vyhx7v5vi/ratings
lee_eisenberg https://www.imdb.com/user/p.f6xwteryvnx326x3jn267izo4u/ratings
jboothmillard https://www.imdb.com/user/p.fwftg73zq35i266aiizvm3fikq/ratings
BA_Harriso

[{'username': 'planktonrules',
  'ratings_url': 'https://www.imdb.com/user/p.twhkadfwuxnnk3v3crk2lbxcda/ratings'},
 {'username': 'SnoopyStyle',
  'ratings_url': 'https://www.imdb.com/user/p.jdpo6uufynnvg5phhb3jhga4ji/ratings'},
 {'username': 'Kirpianuscus',
  'ratings_url': 'https://www.imdb.com/user/p.idvjk25ntslz3t4zlr4bthixra/ratings'},
 {'username': 'mark.waltz',
  'ratings_url': 'https://www.imdb.com/user/p.f3fpx4l7nz7xjpaxcnmbs6mfue/ratings'},
 {'username': 'Sleepin_Dragon',
  'ratings_url': 'https://www.imdb.com/user/p.nadoqibapcutzfhtliuzriogla/ratings'},
 {'username': 'Leofwine_draca',
  'ratings_url': 'https://www.imdb.com/user/p.f772breweshp773tnlaavsxh5m/ratings'},
 {'username': 'Hitchcoc',
  'ratings_url': 'https://www.imdb.com/user/p.bprpogbwb56iaix6ajsznizkde/ratings'},
 {'username': 'paul_m_haakonsen',
  'ratings_url': 'https://www.imdb.com/user/p.jhijfk6m2bldmwgfat3yuhau5m/ratings'},
 {'username': 'ma-cortes',
  'ratings_url': 'https://www.imdb.com/user/p.4ezgnj6y3mutu

# Combine ratings function with user list above

In [ ]:
def user_rev_250(usr):
    #usr will be a dictionary consisting of {'username': 'blah_1',
    #  'ratings_url: 'https://www.imdb.com/user/p.twhkadfwuxnnk3v3crk2lbxcda/ratings'}, e.g
    uname= usr['username']
    url=usr['ratings_url']
    driver = webdriver.Chrome()
    url+="?sort=num_votes%2Cdesc"
    driver.get(url)
    usr_ratings=[]
    my_button = WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "button.ipc-responsive-button"))
)
    ################3#for i in range(4):
    soup=BeautifulSoup(driver.page_source, 'html.parser')
    films = soup.find_all('li', class_="ipc-metadata-list-summary-item"); films
    ##################################
    # for each chunk of 250 films, we get year and title and add to usr_ratings 
    for film in films:
        try:
            rating=film.find_all('span',class_="ipc-rating-star--rating")
            title=film.find_all('h4',class_="ipc-title__text")
            year=film.find('li',class_="ipc-inline-list__item")
            #rating has two entries, first avg then user
            rating_text = rating[1].text.strip()
            #get rid of number before the title
            title_text = title[0].text.split('. ', 1)[1]
            year_text = year.text.strip()
            usr_ratings.append({'title': title_text, 'year': year_text, 'user': uname, 'rating': rating_text})
        except:
            # Skip if either element is not found in this <li>
            continue
    ### IGNORE -- attmpt to load more reviews
    #button = WebDriverWait(driver, 5).until(EC.presence_of_element_located((By.CSS_SELECTOR, "button.ipc-responsive-button")))
        #scroll on page to reach button, otherwise click fails
    #    driver.execute_script("arguments[0].scrollIntoView(true);", button)
    #    sleep(1)        
        # Click it
    #    button.click()
    df=pd.DataFrame(usr_ratings)
    driver.close()
    return df

    

In [ ]:
# create empty data frame to populate
big_df=pd.DataFrame(columns=['title', 'year', 'user', 'rating']); 

In [ ]:
# small scale test to see how code works
for i in range(4):
    try:
        dfu=user_rev_250(usr_links[i])
        print(dfu)
        big_df=pd.concat([big_df,dfu], ignore_index=True)
    except:
        continue

big_df
    

[{'title': 'The Shawshank Redemption', 'year': '1994', 'user': 'planktonrules', 'rating': '10'}, {'title': 'The Dark Knight', 'year': '2008', 'user': 'planktonrules', 'rating': '9'}, {'title': 'Inception', 'year': '2010', 'user': 'planktonrules', 'rating': '8'}, {'title': 'Fight Club', 'year': '1999', 'user': 'planktonrules', 'rating': '6'}, {'title': 'Interstellar', 'year': '2014', 'user': 'planktonrules', 'rating': '8'}, {'title': 'Forrest Gump', 'year': '1994', 'user': 'planktonrules', 'rating': '8'}, {'title': 'Pulp Fiction', 'year': '1994', 'user': 'planktonrules', 'rating': '10'}, {'title': 'The Matrix', 'year': '1999', 'user': 'planktonrules', 'rating': '7'}, {'title': 'The Godfather', 'year': '1972', 'user': 'planktonrules', 'rating': '10'}, {'title': 'The Lord of the Rings: The Fellowship of the Ring', 'year': '2001', 'user': 'planktonrules', 'rating': '10'}, {'title': 'The Lord of the Rings: The Return of the King', 'year': '2003', 'user': 'planktonrules', 'rating': '10'}, {'

,title,year,user,rating
0,The Shawshank Redemption,1994,planktonrules,10
1,The Dark Knight,2008,planktonrules,9
2,Inception,2010,planktonrules,8
3,Fight Club,1999,planktonrules,6
4,Interstellar,2014,planktonrules,8
...,...,...,...,...
745,Ace Ventura: Pet Detective,1994,mark.waltz,1
746,The Untouchables,1987,mark.waltz,9
747,Alien³,1992,mark.waltz,5
748,Moonlight,2016,mark.waltz,10


In [ ]:
# Attempts to get reviews of 250 most popular movies from every user in usr_links
for person in usr_links:
    try:
        dfu=user_rev_250(person)
        print(dfu)
        big_df=pd.concat([big_df,dfu], ignore_index=True)
    except:
        continue

big_df

[{'title': 'The Shawshank Redemption', 'year': '1994', 'user': 'planktonrules', 'rating': '10'}, {'title': 'The Dark Knight', 'year': '2008', 'user': 'planktonrules', 'rating': '9'}, {'title': 'Inception', 'year': '2010', 'user': 'planktonrules', 'rating': '8'}, {'title': 'Fight Club', 'year': '1999', 'user': 'planktonrules', 'rating': '6'}, {'title': 'Interstellar', 'year': '2014', 'user': 'planktonrules', 'rating': '8'}, {'title': 'Forrest Gump', 'year': '1994', 'user': 'planktonrules', 'rating': '8'}, {'title': 'Pulp Fiction', 'year': '1994', 'user': 'planktonrules', 'rating': '10'}, {'title': 'The Matrix', 'year': '1999', 'user': 'planktonrules', 'rating': '7'}, {'title': 'The Godfather', 'year': '1972', 'user': 'planktonrules', 'rating': '10'}, {'title': 'The Lord of the Rings: The Fellowship of the Ring', 'year': '2001', 'user': 'planktonrules', 'rating': '10'}, {'title': 'The Lord of the Rings: The Return of the King', 'year': '2003', 'user': 'planktonrules', 'rating': '10'}, {'

,title,year,user,rating
0,The Shawshank Redemption,1994,planktonrules,10
1,The Dark Knight,2008,planktonrules,9
2,Inception,2010,planktonrules,8
3,Fight Club,1999,planktonrules,6
4,Interstellar,2014,planktonrules,8
...,...,...,...,...
17085,Lady Bird,2017,TaylorYee94,7
17086,Liar Liar,1997,TaylorYee94,5
17087,Les Misérables,2012,TaylorYee94,8
17088,Up in the Air,2009,TaylorYee94,8


In [ ]:
#exportas data frame into csv file
big_df.to_csv('imdb_data.csv')